# Skin Lesion Classification with IBR6Net + Swin Cross-Attention

This notebook follows the updated plan discussed in chat:

- **Stage 1:** train at **224×224**
- **Stage 2:** fine-tune at **384×384**
- CNN branch: **IBR6Net**
- Transformer branch: **Swin-style hierarchical stages**
- Fusion: **cross-attention** at four scales
- Final head: **GAP + concatenate + dense classifier**

The notebook is written to be practical for ISIC-2019 style skin lesion classification.


In [1]:
# Lightweight optional dependency for experiment tracking
!pip -q install wandb


In [2]:
import os
import gc
import math
import glob
from collections import Counter
from dataclasses import dataclass
from typing import Any

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras import layers, Model, Input, mixed_precision
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    TerminateOnNaN,
    BackupAndRestore,
    CSVLogger,
)

import wandb 
from wandb.integration.keras import WandbMetricsLogger
_WANDB_AVAILABLE = True
# except Exception:
#     wandb = None
#     WandbMetricsLogger = None
#     _WANDB_AVAILABLE = False

tf.keras.utils.set_random_seed(42)
tf.keras.backend.clear_session()

# Mixed precision: float16 compute with float32 variables for numerical stability
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

AUTOTUNE = tf.data.AUTOTUNE
print("TensorFlow version:", tf.__version__)

2026-04-21 17:55:50.250854: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776794150.468589      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776794150.525510      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776794150.999190      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776794150.999231      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776794150.999233      55 computation_placer.cc:177] computation placer alr

Mixed precision policy: <DTypePolicy "mixed_float16">
TensorFlow version: 2.19.0


## 0) Optional W&B setup

In [3]:
USE_WANDB = os.getenv("USE_WANDB", "1").strip().lower() in {"1", "true", "yes", "y"}

if USE_WANDB and not _WANDB_AVAILABLE:
    print("W&B was requested but not available, switching to local mode.")
    USE_WANDB = False


In [4]:
# -------------------------------------------------------
# Configuration
# -------------------------------------------------------
class ConfigDict(dict):
    def __getattr__(self, key: str) -> Any:
        try:
            return self[key]
        except KeyError as e:
            raise AttributeError(key) from e

    def __setattr__(self, key: str, value: Any) -> None:
        self[key] = value

model_name = "ibr6_swin_crossattention_two_stage"
run_name = f"isic2019_{model_name}"

default_config = {
    "seed": 42,
    "num_classes": 0,

    # Stage 1
    "stage1_img_size": 224,
    "stage1_epochs": 1,
    "stage1_batch_size": 16,
    "stage1_lr": 1e-4,

    # Stage 2
    "stage2_img_size": 384,
    "stage2_epochs": 1,
    "stage2_batch_size": 4,
    "stage2_lr": 1e-5,

    # Model
    "stem_channels": 64,
    "fusion_dims": (192, 384, 768, 1536),
    "swin_embed_dim": 192,
    "swin_heads": 8,
    "fusion_heads": 8,
    "expansion": 3,
    "dropout_rate": 0.4,

    # Swin windows
    "stage1_window_size": 7,
    "stage2_window_size": 12,

    # Regularization
    "weight_decay": 1e-4,

    # W&B
    "wandb_project": "Proposed_Model",
    "wandb_entity": "",
    "wandb_model_artifact": f"{model_name}-model",

    # Local dataset folders
    "dataset_dir": "./dataset",
    "train_dir": "",
    "val_dir": "",

    "model_name": model_name,
    "run_name": run_name,
}
cfg = ConfigDict(default_config)
cfg.train_dir = os.path.join(cfg.dataset_dir, "train")
cfg.val_dir = os.path.join(cfg.dataset_dir, "validation")

print(cfg)

tf.keras.utils.set_random_seed(cfg.seed)
np.random.seed(cfg.seed)
print("Seed set to:", cfg.seed)

{'seed': 42, 'num_classes': 0, 'stage1_img_size': 224, 'stage1_epochs': 50, 'stage1_batch_size': 8, 'stage1_lr': 0.0001, 'stage2_img_size': 384, 'stage2_epochs': 25, 'stage2_batch_size': 4, 'stage2_lr': 1e-05, 'stem_channels': 64, 'fusion_dims': (192, 384, 768, 1536), 'swin_embed_dim': 192, 'swin_heads': 8, 'fusion_heads': 8, 'expansion': 3, 'dropout_rate': 0.4, 'stage1_window_size': 7, 'stage2_window_size': 12, 'weight_decay': 0.0001, 'wandb_project': 'Proposed_Model', 'wandb_entity': '', 'wandb_model_artifact': 'ibr6_swin_crossattention_two_stage-model', 'dataset_dir': './dataset', 'train_dir': './dataset/train', 'val_dir': './dataset/validation', 'model_name': 'ibr6_swin_crossattention_two_stage', 'run_name': 'isic2019_ibr6_swin_crossattention_two_stage'}
Seed set to: 42


In [5]:
# GPU tracking and GPU-specific batch tuning removed for a cleaner notebook.
# TensorFlow will use the available hardware automatically.

## 1) Load dataset

In [6]:
if USE_WANDB and wandb is not None:
    wandb.login(key="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")
    init_kwargs = dict(
        project=cfg.wandb_project,
        job_type="train",
        name=cfg.run_name,
        config=dict(cfg),
    )
    if cfg.wandb_entity:
        init_kwargs["entity"] = cfg.wandb_entity

    run = wandb.init(**init_kwargs)
else:
    run = None

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: fth830564 (fth830564-f-t-g-haulage) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
dataset_dir = None

artifact = run.use_artifact(f"fth830564-f-t-g-haulage/dataset-pipeline-isic2019/isic2019-full-fixed-dataset:v0", type="dataset")
dataset_dir = artifact.download()



wandb: Downloading large artifact 'isic2019-full-fixed-dataset:v0', 1163.94MB. 25336 files...
wandb:   25336 of 25336 files downloaded.  
Done. 00:02:08.3 (9.1MB/s)


In [8]:
dataset_dir="/kaggle/working/artifacts/isic2019-full-fixed-dataset:v0"

In [9]:
# Build train/val paths safely
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")
test_dir = os.path.join(dataset_dir, "test")


In [10]:
train_dir

'/kaggle/working/artifacts/isic2019-full-fixed-dataset:v0/train'

In [11]:
# import os

# Ensure dataset_dir exists
# dataset_dir = os.path.abspath(cfg.dataset_dir)


# ---- Validate TRAIN directory ----
if not os.path.isdir(train_dir):
    raise FileNotFoundError(f"❌ Training folder not found: {train_dir}")

# ---- Validate VALIDATION directory ----
if not os.path.isdir(val_dir):
    fallback_val = os.path.join(dataset_dir, "val")

    if os.path.isdir(fallback_val):
        print("⚠️ Using fallback validation folder:", fallback_val)
        val_dir = fallback_val
    else:
        raise FileNotFoundError(
            f"❌ Validation folder not found:\n{val_dir}\nOR\n{fallback_val}"
        )

# ---- Update config ----
cfg.dataset_dir = dataset_dir
cfg.train_dir = train_dir
cfg.val_dir = val_dir

# ---- Print info ----
print("📁 Dataset directory:", cfg.dataset_dir)
print("📁 Train directory:", cfg.train_dir)
print("📁 Validation directory:", cfg.val_dir)

# ---- List classes safely ----
classes = sorted([
    d for d in os.listdir(cfg.train_dir)
    if os.path.isdir(os.path.join(cfg.train_dir, d))
])

print("📊 Train classes:", classes)
print("📊 Number of classes:", len(classes))

📁 Dataset directory: /kaggle/working/artifacts/isic2019-full-fixed-dataset:v0
📁 Train directory: /kaggle/working/artifacts/isic2019-full-fixed-dataset:v0/train
📁 Validation directory: /kaggle/working/artifacts/isic2019-full-fixed-dataset:v0/val
📊 Train classes: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']
📊 Number of classes: 8


In [12]:
VALID_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif"}

data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
        layers.RandomTranslation(0.10, 0.10),
    ],
    name="training_augmentation",
)

def count_images_in_class(root_dir: str, class_name: str) -> int:
    class_dir = os.path.join(root_dir, class_name)
    if not os.path.isdir(class_dir):
        return 0

    total = 0
    for file_path in glob.glob(os.path.join(class_dir, "**", "*"), recursive=True):
        if os.path.isfile(file_path) and os.path.splitext(file_path.lower())[1] in VALID_IMAGE_EXTENSIONS:
            total += 1
    return total

def make_image_dataset(directory: str, image_size: int, batch_size: int, training: bool):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        image_size=(image_size, image_size),
        batch_size=batch_size,
        shuffle=training,
        seed=cfg.seed if training else None,
    )

def preprocess_batch(images, labels, training=False):
    images = tf.cast(images, tf.float16) / 255.0
    if training:
        images = data_augmentation(images, training=True)
    labels = tf.cast(labels, tf.int32)
    return images, labels

I0000 00:00:1776794324.430840      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776794324.436972      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [13]:
train_ds_224_raw = make_image_dataset(cfg.train_dir, cfg.stage1_img_size, cfg.stage1_batch_size, training=True)
val_ds_224_raw = make_image_dataset(cfg.val_dir, cfg.stage1_img_size, cfg.stage1_batch_size, training=False)

class_names = list(train_ds_224_raw.class_names)
cfg.num_classes = len(class_names)

train_ds_224 = train_ds_224_raw.map(
    lambda images, labels: preprocess_batch(images, labels, training=True),
    num_parallel_calls=AUTOTUNE,
).prefetch(1)

val_ds_224 = val_ds_224_raw.map(
    lambda images, labels: preprocess_batch(images, labels, training=False),
    num_parallel_calls=AUTOTUNE,
).prefetch(1)

del train_ds_224_raw, val_ds_224_raw

print("Class names:", class_names)
print("Number of classes:", cfg.num_classes)

Found 20264 files belonging to 8 classes.
Found 2533 files belonging to 8 classes.
Class names: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']
Number of classes: 8


## 2) Data pipeline

In [14]:
class_counts = {idx: count_images_in_class(cfg.train_dir, class_name) for idx, class_name in enumerate(class_names)}
if any(count <= 0 for count in class_counts.values()):
    missing = [class_names[i] for i, count in class_counts.items() if count <= 0]
    raise ValueError(f"One or more class folders are empty or missing images: {missing}")

total_samples = sum(class_counts.values())
class_weight = {
    idx: float(total_samples / (len(class_counts) * count))
    for idx, count in class_counts.items()
}

alpha_vec = np.array(
    [total_samples / (cfg.num_classes * class_counts[idx]) for idx in range(cfg.num_classes)],
    dtype=np.float32,
)
alpha_vec = alpha_vec / np.sum(alpha_vec) * cfg.num_classes

print("Class counts:", class_counts)
print("Class weight:", class_weight)
print("Alpha vector:", alpha_vec)

Class counts: {0: 693, 1: 2659, 2: 2099, 3: 191, 4: 3618, 5: 10299, 6: 502, 7: 203}
Class weight: {0: 3.655122655122655, 1: 0.9526137645731478, 2: 1.2067651262505956, 3: 13.261780104712042, 4: 0.7001105583195135, 5: 0.24594620836974462, 6: 5.0458167330677295, 7: 12.47783251231527}
Alpha vector: [0.7788044  0.20297535 0.25712788 2.8257143  0.14917397 0.05240426
 1.0751224  2.658677  ]


## 3) Model blocks

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

In [15]:
# =========================
# IBRBlock(CNN)
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class IBRBlock(layers.Layer):
    def __init__(self, filters, stride=1, expansion=4, se_ratio=0.25, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.stride = stride
        self.expansion = expansion
        self.se_ratio = se_ratio
        self.dropout = dropout

        self.expand_conv = None
        self.expand_bn = None
        self.dw_conv = None
        self.dw_bn = None
        self.project_conv = None
        self.project_bn = None
        self.drop = layers.Dropout(dropout)

        self.shortcut_proj = None
        self.shortcut_bn = None

        self.act = layers.Activation("swish")

    def build(self, input_shape):
        in_channels = int(input_shape[-1])
        hidden_dim = in_channels * self.expansion

        self.expand_conv = layers.Conv2D(hidden_dim, 1, padding="same", use_bias=False)
        self.expand_bn = layers.BatchNormalization()

        self.dw_conv = layers.DepthwiseConv2D(
            3, strides=self.stride, padding="same", use_bias=False
        )
        self.dw_bn = layers.BatchNormalization()

        se_hidden = max(8, int(hidden_dim * self.se_ratio))
        self.se_gap = layers.GlobalAveragePooling2D()
        self.se_fc1 = layers.Dense(se_hidden, activation="swish")
        self.se_fc2 = layers.Dense(hidden_dim, activation="sigmoid")

        self.project_conv = layers.Conv2D(self.filters, 1, padding="same", use_bias=False)
        self.project_bn = layers.BatchNormalization()

        if self.stride != 1 or in_channels != self.filters:
            self.shortcut_proj = layers.Conv2D(self.filters, 1, strides=self.stride, padding="same", use_bias=False)
            self.shortcut_bn = layers.BatchNormalization()

        super().build(input_shape)

    def call(self, x, training=None):
        shortcut = x

        x = self.expand_conv(x)
        x = self.expand_bn(x, training=training)
        x = self.act(x)

        x = self.dw_conv(x)
        x = self.dw_bn(x, training=training)
        x = self.act(x)

        se = self.se_gap(x)
        se = self.se_fc1(se)
        se = self.se_fc2(se)
        se = tf.reshape(se, [-1, 1, 1, tf.shape(x)[-1]])
        x = x * se

        x = self.project_conv(x)
        x = self.project_bn(x, training=training)

        if self.shortcut_proj is not None:
            shortcut = self.shortcut_proj(shortcut)
            shortcut = self.shortcut_bn(shortcut, training=training)

        x = layers.Add()([x, shortcut])
        x = self.act(x)
        x = self.drop(x, training=training)
        return x



In [ ]:
# =========================
# Swin Block
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class SwinBlock(layers.Layer):
    def __init__(self, dim, num_heads=8, window_size=7, shift_size=0, mlp_ratio=4, dropout=0.0, **kwargs):
        super().__init__(**kwargs)

        if dim % num_heads != 0:
            raise ValueError(f"dim={dim} must be divisible by num_heads={num_heads}")

        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.qkv = layers.Dense(dim * 3, use_bias=False)
        self.proj = layers.Dense(dim)
        self.attn_drop = layers.Dropout(dropout)
        self.proj_drop = layers.Dropout(dropout)

        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(dim * mlp_ratio, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(dim),
        ])

    def build(self, input_shape):
        self.relative_position_bias_table = self.add_weight(
            shape=((2 * self.window_size - 1) * (2 * self.window_size - 1), self.num_heads),
            initializer="zeros",
            trainable=True,
            name="rel_pos_bias"
        )

        coords_h = tf.range(self.window_size)
        coords_w = tf.range(self.window_size)
        coords = tf.stack(tf.meshgrid(coords_h, coords_w, indexing="ij"))
        coords_flatten = tf.reshape(coords, (2, -1))

        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = tf.transpose(relative_coords, [1, 2, 0])

        relative_coords = relative_coords + self.window_size - 1
        relative_coords = relative_coords[:, :, 0] * (2 * self.window_size - 1) + relative_coords[:, :, 1]

        self.relative_position_index = tf.Variable(
            initial_value=tf.cast(relative_coords, tf.int32),
            trainable=False,
            name="rel_pos_index"
        )

        super().build(input_shape)

    def window_partition(self, x):
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C = tf.shape(x)[3]
        ws = self.window_size

        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws

        x = tf.pad(x, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])

        Hp = tf.shape(x)[1]
        Wp = tf.shape(x)[2]

        x = tf.reshape(x, [B, Hp // ws, ws, Wp // ws, ws, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        windows = tf.reshape(x, [-1, ws * ws, C])

        return windows, B, Hp, Wp

    def window_reverse(self, windows, B, Hp, Wp):
        ws = self.window_size
        C = tf.shape(windows)[-1]

        x = tf.reshape(windows, [B, Hp // ws, Wp // ws, ws, ws, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        x = tf.reshape(x, [B, Hp, Wp, C])
        return x

    def call(self, inputs, training=None):
        shortcut = inputs
        x = self.norm1(inputs)

        if self.shift_size > 0:
            x = tf.roll(x, shift=[-self.shift_size, -self.shift_size], axis=[1, 2])

        windows, B, Hp, Wp = self.window_partition(x)

        qkv = self.qkv(windows)
        qkv = tf.reshape(qkv, [-1, tf.shape(windows)[1], 3, self.num_heads, self.dim // self.num_heads])
        qkv = tf.transpose(qkv, [2, 0, 3, 1, 4])

        q, k, v = qkv[0], qkv[1], qkv[2]

        scale = (self.dim // self.num_heads) ** -0.5
        attn = tf.matmul(q, k, transpose_b=True) * scale

        bias = tf.gather(self.relative_position_bias_table,
                         tf.reshape(self.relative_position_index, [-1]))
        bias = tf.reshape(bias, [self.window_size * self.window_size,
                                 self.window_size * self.window_size,
                                 self.num_heads])
        bias = tf.transpose(bias, [2, 0, 1])

        attn = attn + tf.expand_dims(bias, 0)
        attn = tf.nn.softmax(attn, axis=-1)
        attn = self.attn_drop(attn, training=training)

        out = tf.matmul(attn, v)
        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, [-1, self.window_size * self.window_size, self.dim])

        out = self.proj(out)
        out = self.proj_drop(out, training=training)

        x = self.window_reverse(out, B, Hp, Wp)

        if self.shift_size > 0:
            x = tf.roll(x, shift=[self.shift_size, self.shift_size], axis=[1, 2])

        x = x[:, :tf.shape(shortcut)[1], :tf.shape(shortcut)[2], :]
        x = shortcut + x
        x = x + self.mlp(self.norm2(x), training=training)

        return x



In [16]:



# =========================
# Patch Embedding
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class PatchEmbed(layers.Layer):
    def __init__(self, embed_dim=192, patch_size=4, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.patch_size = patch_size

        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation("swish")
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=None):
        x = self.proj(x)
        x = self.bn(x, training=training)
        x = self.act(x)
        x = self.norm(x)
        return x


# =========================
# Patch Merging
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class PatchMerging(layers.Layer):
    def __init__(self, out_dim, **kwargs):
        super().__init__(**kwargs)
        self.out_dim = out_dim
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.reduction = None

    def build(self, input_shape):
        self.reduction = layers.Dense(self.out_dim, use_bias=False)
        super().build(input_shape)

    def call(self, x):
        # x: (B, H, W, C)
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C = tf.shape(x)[3]

        pad_h = H % 2
        pad_w = W % 2
        x = tf.pad(x, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])

        H2 = tf.shape(x)[1]
        W2 = tf.shape(x)[2]

        x0 = x[:, 0:H2:2, 0:W2:2, :]
        x1 = x[:, 1:H2:2, 0:W2:2, :]
        x2 = x[:, 0:H2:2, 1:W2:2, :]
        x3 = x[:, 1:H2:2, 1:W2:2, :]

        x = tf.concat([x0, x1, x2, x3], axis=-1)
        x = self.norm(x)
        x = self.reduction(x)
        return x



In [ ]:
# =========================
# Swin Stage
# =========================
def swin_stage(x, dim, depth, num_heads, window_size=7, dropout=0.0, name_prefix="swin"):
    for i in range(depth):
        shift = 0 if i % 2 == 0 else window_size // 2
        x = SwinBlock(
            dim=dim,
            num_heads=num_heads,
            window_size=window_size,
            shift_size=shift,
            mlp_ratio=4,
            dropout=dropout,
            name=f"{name_prefix}_block_{i+1}"
        )(x)
    return x



In [ ]:
# =========================
# Bi-Directional Cross Attention Fusion
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class BiDirectionalCrossAttentionFusion(layers.Layer):
    def __init__(self, fusion_dim, num_heads=8, mlp_ratio=4, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.fusion_dim = fusion_dim
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout

        self.cnn_proj = layers.Conv2D(fusion_dim, 1, padding="same", use_bias=False)
        self.swin_proj = layers.Conv2D(fusion_dim, 1, padding="same", use_bias=False)

        self.cnn_bn = layers.BatchNormalization()
        self.swin_bn = layers.BatchNormalization()

        self.norm_c = layers.LayerNormalization(epsilon=1e-6)
        self.norm_s = layers.LayerNormalization(epsilon=1e-6)
        self.norm_f = layers.LayerNormalization(epsilon=1e-6)

        self.attn_c_to_s = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=fusion_dim // num_heads, dropout=dropout, name="attn_c_to_s"
        )
        self.attn_s_to_c = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=fusion_dim // num_heads, dropout=dropout, name="attn_s_to_c"
        )

        self.drop = layers.Dropout(dropout)

        self.ffn = tf.keras.Sequential([
            layers.Dense(fusion_dim * mlp_ratio, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(fusion_dim),
        ])

        self.gate = layers.Dense(fusion_dim, activation="sigmoid")

    def _resize_if_needed(self, x, target_hw):
        th, tw = target_hw
        xh = tf.shape(x)[1]
        xw = tf.shape(x)[2]
        return tf.cond(
            tf.logical_or(tf.not_equal(xh, th), tf.not_equal(xw, tw)),
            lambda: tf.image.resize(x, (th, tw), method="bilinear"),
            lambda: x
        )

    def call(self, inputs, training=None):
        cnn_feat, swin_feat = inputs  # (B,H,W,C1), (B,H,W,C2)

        target_hw = (tf.shape(cnn_feat)[1], tf.shape(cnn_feat)[2])
        swin_feat = self._resize_if_needed(swin_feat, target_hw)

        cnn_feat = self.cnn_proj(cnn_feat)
        cnn_feat = self.cnn_bn(cnn_feat, training=training)

        swin_feat = self.swin_proj(swin_feat)
        swin_feat = self.swin_bn(swin_feat, training=training)

        B = tf.shape(cnn_feat)[0]
        H = tf.shape(cnn_feat)[1]
        W = tf.shape(cnn_feat)[2]

        cnn_tokens = tf.reshape(cnn_feat, [B, H * W, self.fusion_dim])
        swin_tokens = tf.reshape(swin_feat, [B, H * W, self.fusion_dim])

        cnn_tokens_n = self.norm_c(cnn_tokens)
        swin_tokens_n = self.norm_s(swin_tokens)

        # Bi-directional cross attention
        cnn_to_swin = self.attn_c_to_s(query=cnn_tokens_n, value=swin_tokens_n, key=swin_tokens_n, training=training)
        swin_to_cnn = self.attn_s_to_c(query=swin_tokens_n, value=cnn_tokens_n, key=cnn_tokens_n, training=training)

        cnn_tokens = cnn_tokens + self.drop(cnn_to_swin, training=training)
        swin_tokens = swin_tokens + self.drop(swin_to_cnn, training=training)

        # Fuse both directions
        fused = tf.concat([cnn_tokens, swin_tokens], axis=-1)
        fused = layers.Dense(self.fusion_dim)(fused)
        fused = fused + self.ffn(self.norm_f(fused), training=training)

        # Gated refinement
        gate = self.gate(fused)
        fused = fused * gate + fused

        fused = tf.reshape(fused, [B, H, W, self.fusion_dim])
        return fused



In [ ]:
class FeatureAttentionFusion(layers.Layer):
    def __init__(self, hidden_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim

    def build(self, input_shape):
        # Create separate projection layers for each feature
        self.projections = []
        self.score_layers = []

        for shape in input_shape:
            self.projections.append(
                layers.Dense(self.hidden_dim, activation="swish")
            )
            self.score_layers.append(
                layers.Dense(1)
            )

    def call(self, features):
        scores = []
        projected_features = []

        # Compute scores independently
        for i, f in enumerate(features):
            p = self.projections[i](f)      # project to same dim
            s = self.score_layers[i](p)     # (B,1)

            projected_features.append(f)    # keep original
            scores.append(s)

        scores = tf.concat(scores, axis=1)        # (B, N)
        weights = tf.nn.softmax(scores, axis=1)   # (B, N)

        weighted_features = []
        for i, f in enumerate(projected_features):
            w = tf.expand_dims(weights[:, i], axis=-1)
            weighted_features.append(f * w)

        return tf.concat(weighted_features, axis=-1)

In [ ]:
def build_model(
    input_shape=(224, 224, 3),
    num_classes=7,
    stem_channels=64,
    swin_embed_dim=192,
    swin_heads=8,
    fusion_heads=8,
    expansion=4,
    dropout_rate=0.3,
    window_size=7
):
    inputs = Input(shape=input_shape, name="input_image")

    # -----------------
    # CNN branch
    # -----------------
    x = layers.Conv2D(stem_channels, 7, strides=2, padding="same",
                      use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("swish", name="stem_swish")(x)

    f1 = IBRBlock(64, stride=1, expansion=expansion, dropout=0.0, name="ibr_block_1")(x)
    f2 = IBRBlock(128, stride=2, expansion=expansion, dropout=0.0, name="ibr_block_2")(f1)
    f3 = IBRBlock(256, stride=2, expansion=expansion, dropout=0.0, name="ibr_block_3")(f2)
    f4 = IBRBlock(512, stride=2, expansion=expansion, dropout=0.0, name="ibr_block_4")(f3)

    # -----------------
    # Swin branch
    # -----------------
    s = PatchEmbed(embed_dim=swin_embed_dim, name="patch_embed")(inputs)

    s1 = swin_stage(s, dim=swin_embed_dim, depth=2,
                    num_heads=swin_heads, window_size=window_size,
                    dropout=0.0, name_prefix="swin_stage1")

    s = PatchMerging(out_dim=swin_embed_dim * 2, name="patch_merge_1")(s1)

    s2 = swin_stage(s, dim=swin_embed_dim * 2, depth=2,
                    num_heads=swin_heads, window_size=window_size,
                    dropout=0.0, name_prefix="swin_stage2")

    s = PatchMerging(out_dim=swin_embed_dim * 4, name="patch_merge_2")(s2)

    s3 = swin_stage(s, dim=swin_embed_dim * 4, depth=2,
                    num_heads=swin_heads, window_size=window_size,
                    dropout=0.0, name_prefix="swin_stage3")

    # -----------------
    # Fusion stages
    # -----------------
    o1 = BiDirectionalCrossAttentionFusion(
        fusion_dim=192, num_heads=fusion_heads,
        mlp_ratio=4, dropout=0.1,
        name="fusion_stage_1"
    )([f2, s1])

    o2 = BiDirectionalCrossAttentionFusion(
        fusion_dim=384, num_heads=fusion_heads,
        mlp_ratio=4, dropout=0.1,
        name="fusion_stage_2"
    )([f3, s2])

    o3 = BiDirectionalCrossAttentionFusion(
        fusion_dim=512, num_heads=fusion_heads,
        mlp_ratio=4, dropout=0.1,
        name="fusion_stage_3"
    )([f4, s3])

    # -----------------
    # Global Pooling
    # -----------------
    gap_o1 = layers.GlobalAveragePooling2D(name="gap_o1")(o1)
    gap_o2 = layers.GlobalAveragePooling2D(name="gap_o2")(o2)
    gap_o3 = layers.GlobalAveragePooling2D(name="gap_o3")(o3)

    # -----------------
    # Feature Projection
    # -----------------
    gap_o1 = layers.Dense(256, name="proj_o1")(gap_o1)
    gap_o2 = layers.Dense(256, name="proj_o2")(gap_o2)
    gap_o3 = layers.Dense(256, name="proj_o3")(gap_o3)

    gap_o1 = layers.LayerNormalization(name="ln_o1")(gap_o1)
    gap_o2 = layers.LayerNormalization(name="ln_o2")(gap_o2)
    gap_o3 = layers.LayerNormalization(name="ln_o3")(gap_o3)

    # -----------------
    # Attention Fusion
    # -----------------
    fusion = FeatureAttentionFusion(name="feature_attention")(
        [gap_o1, gap_o2, gap_o3]
    )

    fusion = layers.LayerNormalization(name="fusion_ln")(fusion)
    fusion = layers.Dropout(0.2, name="fusion_dropout")(fusion)

    # -----------------
    # Residual MLP Head
    # -----------------
    res = layers.Dense(512, activation="swish", name="mlp_dense_1")(fusion)
    x = layers.Dense(512, activation="swish", name="mlp_dense_2")(res)
    x = layers.Add(name="mlp_residual_add")([x, res])
    x = layers.Dropout(dropout_rate, name="mlp_dropout")(x)

    # -----------------
    # Output
    # -----------------
    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions"
    )(x)

    return Model(inputs=inputs, outputs=outputs, name="fused_IBR4Net_Swin_T")

## 4) Build the full model

In [17]:
def sparse_focal_loss(num_classes, alpha=None, gamma=2.0):
    alpha = tf.constant(alpha if alpha is not None else np.ones(num_classes, dtype=np.float32), dtype=tf.float32)

    def loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.cast(y_pred, tf.float32)
        eps = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)

        y_true_one_hot = tf.one_hot(y_true, depth=num_classes)
        pt = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        alpha_t = tf.gather(alpha, y_true)
        return -alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)

    return loss


def get_optimizer(lr, weight_decay):
    try:
        return tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay, clipnorm=1.0)
    except Exception:
        return tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0)


## 5) Stage 1: train at 224×224

In [18]:
tf.keras.backend.clear_session()

model_224 = build_new_model(
    input_shape=(cfg.stage1_img_size, cfg.stage1_img_size, 3),
    num_classes=cfg.num_classes,
    stem_channels=cfg.stem_channels,
    fusion_dims=cfg.fusion_dims,
    swin_embed_dim=cfg.swin_embed_dim,
    swin_heads=cfg.swin_heads,
    fusion_heads=cfg.fusion_heads,
    expansion=cfg.expansion,
    dropout_rate=cfg.dropout_rate,
    window_size=cfg.stage1_window_size,
)

model_224.summary()


Model: "IBR6Net_Swin_CrossAttention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed         │ (None, 56, 56,    │      9,600 │ input_image[0][0] │
│ (PatchEmbed)        │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage1_block_1 │ (None, 56, 56,    │    444,864 │ patch_embed[0][0] │
│ (SwinBlock)         │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │      9,408 │ input_image[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage1_block_2 │ (None, 56, 56,    │    444,864 │ swin_stage1_bloc… │
│ (SwinBlock)         │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_merge_1       │ (None, 28, 28,    │    295,296 │ swin_stage1_bloc… │
│ (PatchMerging)      │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_swish          │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage2_block_1 │ (None, 28, 28,    │  1,774,464 │ patch_merge_1[0]… │
│ (SwinBlock)         │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_1         │ (None, 112, 112,  │     28,096 │ stem_swish[0][0]  │
│ (IBRBlock)          │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage2_block_2 │ (None, 28, 28,    │  1,774,464 │ swin_stage2_bloc… │
│ (SwinBlock)         │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_2         │ (None, 56, 56,    │     80,768 │ ibr_block_1[0][0] │
│ (IBRBlock)          │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_merge_2       │ (None, 14, 14,    │  1,180,416 │ swin_stage2_bloc… │
│ (PatchMerging)      │ 768)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_3         │ (None, 28, 28,    │    308,992 │ ibr_block_2[0][0] │
│ (IBRBlock)          │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage3_block_1 │ (None, 14, 14,    │  7,087,872 │ patch_merge_2[0]… │
│ (SwinBlock)         │ 768)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_4         │ (None, 14, 14,    │  1,207,808 │ ibr_block_3[0][0] │
│ (IBRBlock)          │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage3_block_2 │ (None, 14, 14,    │  7,087,872 │ swin_stage3_bloc

 Total params: 30,561,352 (116.58 MB)

 Trainable params: 30,547,272 (116.53 MB)

 Non-trainable params: 14,080 (55.00 KB)

In [19]:
model_224.compile(
    optimizer=get_optimizer(cfg.stage1_lr, cfg.weight_decay),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
    ],
)

stage1_callbacks = [
    ModelCheckpoint(
        f"{cfg.model_name}_stage1.weights.h5",
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
        mode="max",
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        min_delta=1e-4,
        restore_best_weights=True,
        mode="max",
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    TerminateOnNaN(),
    BackupAndRestore(backup_dir=f"./backup_{cfg.model_name}_stage1"),
    CSVLogger(f"{cfg.model_name}_stage1.csv"),
]

if USE_WANDB and WandbMetricsLogger is not None:
    stage1_callbacks.append(WandbMetricsLogger(log_freq="epoch"))

In [ ]:
history_stage1 = model_224.fit(
    train_ds_224,
    validation_data=val_ds_224,
    epochs=cfg.stage1_epochs,
    class_weight=class_weight,
    callbacks=stage1_callbacks,
    verbose=1,
)


Epoch 1/50


I0000 00:00:1776794366.593967     289 service.cc:152] XLA service 0x7e192c010cc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776794366.594013     289 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776794366.594018     289 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776794375.324687     289 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-21 17:59:45.853263: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 17:59:46.014119: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 17:59:47.040571: E external/local_xl

2533/2533 ━━━━━━━━━━━━━━━━━━━━ 0s 664ms/step - accuracy: 0.2934 - loss: 2.3326 - top2_acc: 0.4605
Epoch 1: val_accuracy improved from -inf to 0.13107, saving model to ibr6_swin_crossattention_two_stage_stage1.weights.h5
2533/2533 ━━━━━━━━━━━━━━━━━━━━ 1862s 695ms/step - accuracy: 0.2934 - loss: 2.3325 - top2_acc: 0.4605 - val_accuracy: 0.1311 - val_loss: 13.5044 - val_top2_acc: 0.1544 - learning_rate: 1.0000e-04
Epoch 2/50
 771/2533 ━━━━━━━━━━━━━━━━━━━━ 19:34 667ms/step - accuracy: 0.4379 - loss: 1.8538 - top2_acc: 0.6167

In [ ]:
# Load the best stage 1 weights
stage1_weights_path = f"{cfg.model_name}_stage1.weights.h5"
model_224.load_weights(stage1_weights_path)
print("Loaded stage 1 weights:", stage1_weights_path)

# Release stage 1 objects before building the 384px model.
del history_stage1
del train_ds_224, val_ds_224
del model_224
del stage1_callbacks
gc.collect()
tf.keras.backend.clear_session()

## 6) Stage 2: fine-tune at 384×384

In [ ]:
train_ds_384_raw = make_image_dataset(cfg.train_dir, cfg.stage2_img_size, cfg.stage2_batch_size, training=True)
val_ds_384_raw = make_image_dataset(cfg.val_dir, cfg.stage2_img_size, cfg.stage2_batch_size, training=False)

train_ds_384 = train_ds_384_raw.map(
    lambda images, labels: preprocess_batch(images, labels, training=True),
    num_parallel_calls=AUTOTUNE,
).prefetch(1)

val_ds_384 = val_ds_384_raw.map(
    lambda images, labels: preprocess_batch(images, labels, training=False),
    num_parallel_calls=AUTOTUNE,
).prefetch(1)

del train_ds_384_raw, val_ds_384_raw

tf.keras.backend.clear_session()

model_384 = build_new_model(
    input_shape=(cfg.stage2_img_size, cfg.stage2_img_size, 3),
    num_classes=cfg.num_classes,
    stem_channels=cfg.stem_channels,
    fusion_dims=cfg.fusion_dims,
    swin_embed_dim=cfg.swin_embed_dim,
    swin_heads=cfg.swin_heads,
    fusion_heads=cfg.fusion_heads,
    expansion=cfg.expansion,
    dropout_rate=cfg.dropout_rate,
    window_size=cfg.stage2_window_size,
)

# Transfer weights from the 224 model.
model_384.build((None, cfg.stage2_img_size, cfg.stage2_img_size, 3))
model_384.load_weights(stage1_weights_path)
print("Transferred stage 1 weights into the 384 model.")

In [ ]:
focal_loss = sparse_focal_loss(cfg.num_classes, alpha=alpha_vec, gamma=2.0)

model_384.compile(
    optimizer=get_optimizer(cfg.stage2_lr, cfg.weight_decay),
    loss=focal_loss,
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
    ],
)

stage2_callbacks = [
    ModelCheckpoint(
        f"{cfg.model_name}_final.weights.h5",
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
        mode="max",
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        min_delta=1e-4,
        restore_best_weights=True,
        mode="max",
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    TerminateOnNaN(),
    BackupAndRestore(backup_dir=f"./backup_{cfg.model_name}_stage2"),
    CSVLogger(f"{cfg.model_name}_stage2.csv"),
]

if USE_WANDB and WandbMetricsLogger is not None:
    stage2_callbacks.append(WandbMetricsLogger(log_freq="epoch"))

history_stage2 = model_384.fit(
    train_ds_384,
    validation_data=val_ds_384,
    epochs=cfg.stage2_epochs,
    class_weight=class_weight,
    callbacks=stage2_callbacks,
    verbose=1,
)

## 7) Save the final model

In [ ]:
final_weights_path = f"{cfg.model_name}_final.weights.h5"
model_384.load_weights(final_weights_path)

# Save the full model for later reuse.
final_model_path = f"{cfg.model_name}_final.keras"
model_384.save(final_model_path)
print("Saved final model to:", final_model_path)

if USE_WANDB and wandb is not None and run is not None:
    model_artifact = wandb.Artifact(
        name=cfg.wandb_model_artifact,
        type="model",
        metadata=dict(cfg),
    )
    model_artifact.add_file(final_model_path)
    model_artifact.add_file(final_weights_path)
    model_artifact.add_file(stage1_weights_path)

    if os.path.exists(f"{cfg.model_name}_stage1.csv"):
        model_artifact.add_file(f"{cfg.model_name}_stage1.csv")
    if os.path.exists(f"{cfg.model_name}_stage2.csv"):
        model_artifact.add_file(f"{cfg.model_name}_stage2.csv")

    run.log_artifact(model_artifact, aliases=["latest", "best"])
    run.summary["final_model_path"] = final_model_path
    run.summary["final_weights_path"] = final_weights_path
    run.summary["stage1_weights_path"] = stage1_weights_path


## 8) Evaluation helpers

In [ ]:
def predict_dataset(model, ds):
    y_true = []
    y_pred = []

    for images, labels in ds:
        probs = model(images, training=False)
        preds = tf.argmax(probs, axis=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(preds.numpy().tolist())

    return np.array(y_true), np.array(y_pred)


def plot_confusion_matrix(model, ds, class_names, normalize=True, title="Confusion Matrix"):
    y_true, y_pred = predict_dataset(model, ds)
    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    plt.figure(figsize=(11, 9))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title + (" (Normalized)" if normalize else ""))
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm[i, j]
            txt = f"{value:.2f}" if normalize else str(int(value))
            plt.text(j, i, txt, ha="center", va="center", fontsize=8)

    plt.tight_layout()
    plt.show()

    return y_true, y_pred


def print_classification_report(model, ds, class_names):
    y_true, y_pred = predict_dataset(model, ds)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    return y_true, y_pred


In [ ]:
# Evaluate on the 384 pipeline using the best final weights.
y_true, y_pred = predict_dataset(model_384, val_ds_384)
val_acc = float(np.mean(y_true == y_pred))
print("Validation accuracy:", val_acc)

print_classification_report(model_384, val_ds_384, class_names)
plot_confusion_matrix(model_384, val_ds_384, class_names, normalize=True, title="Validation Confusion Matrix")

if USE_WANDB and wandb is not None and run is not None:
    run.summary["validation_accuracy"] = val_acc

## 9) Single-image prediction and TTA

In [ ]:
def _tta_variants(img):
    return [
        img,
        tf.image.flip_left_right(img),
        tf.image.rot90(img, k=1),
        tf.image.rot90(img, k=2),
    ]


def predict_with_tta(image_path, model, image_size=384):
    img_bytes = tf.io.read_file(image_path)
    img = tf.io.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, (image_size, image_size), method="bilinear")
    img = tf.cast(img, tf.float16) / 255.0

    preds = []
    for aug in _tta_variants(img):
        aug = tf.expand_dims(aug, axis=0)
        pred = model.predict(aug, verbose=0)[0]
        preds.append(pred)

    pred = np.mean(np.stack(preds, axis=0), axis=0)
    pred_idx = int(np.argmax(pred))
    pred_name = class_names[pred_idx]

    print("Pred idx:", pred_idx)
    print("Pred name:", pred_name)
    print("Probabilities:", pred)
    return pred, pred_name


# Example:
# example_image_path = tf.io.gfile.glob(os.path.join(cfg.val_dir, "*", "*"))[0]
# predict_with_tta(example_image_path, model_384, image_size=cfg.stage2_img_size)

## 10) Finish

In [ ]:
if USE_WANDB and wandb is not None and run is not None:
    wandb.finish()

gc.collect()
